# Payroll Anomaly Detection — Use Case #6

Mendeteksi **anomali penggajian** (salah input, tunjangan negatif, lembur
ekstrem, total tidak konsisten) tanpa harus punya data berlabel.

> **Data:** `data/hr_payroll.csv` **sintetis** (`scripts/generate_payroll_data.py`).
> Kolom `is_anomaly` & `jenis_anomali` adalah **label buatan** yang hanya dipakai
> untuk mengevaluasi — di dunia nyata label ini jarang ada.

## Dua pendekatan yang dibandingkan

1. **Unsupervised** — `IsolationForest` & `LocalOutlierFactor` (tanpa label).
2. **Supervised** — `RandomForest` (memakai label; jadi batas atas/baseline).
3. **Aturan sederhana** — z-score per kolom (pembanding naif).

Karena anomali hanya ~4%, metrik utama **PR-AUC** dan **precision@k**, bukan akurasi.

## 0. Import & Konfigurasi

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.neighbors import LocalOutlierFactor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_score, recall_score, f1_score,
    precision_recall_curve, confusion_matrix,
)

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

CANDIDATE_PATHS = ["data/hr_payroll.csv", "../data/hr_payroll.csv", "hr_payroll.csv"]
DATA_PATH = next((Path(p) for p in CANDIDATE_PATHS if Path(p).exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("hr_payroll.csv tidak ditemukan.")

CWD = Path.cwd()
PROJECT_ROOT = CWD if (CWD / "data").exists() else CWD.parent
ARTIFACT_DIR = PROJECT_ROOT / "models" / "payroll_anomaly" / "v1"

LABEL = "is_anomaly"
LEAKAGE = ["employee_id", "jenis_anomali", LABEL]

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 170)
print("Data :", DATA_PATH)
print("Artefak:", ARTIFACT_DIR)

Data : data/hr_payroll.csv
Artefak: /workspace/project/hr-intelligence-system/models/payroll_anomaly/v1


## 1. Load & Feature Engineering

In [2]:
df = pd.read_csv(DATA_PATH)

# Rasio turunan yang menonjolkan anomali (dihitung aman, tanpa bagi nol).
df["rasio_tunjangan"] = df["tunjangan"] / df["gaji_pokok"].replace(0, np.nan)
df["rasio_potongan"] = df["potongan"] / df["gaji_pokok"].replace(0, np.nan)
df["total_per_pokok"] = df["total_gaji"] / df["gaji_pokok"].replace(0, np.nan)
df["biaya_lembur"] = df["lembur_jam"] * df["tarif_lembur"]

BASE_NUM = ["gaji_pokok", "tunjangan", "lembur_jam", "tarif_lembur", "potongan", "total_gaji"]
DERIVED = ["rasio_tunjangan", "rasio_potongan", "total_per_pokok", "biaya_lembur"]
NUM_COLS = BASE_NUM + DERIVED
CAT_COLS = ["departemen", "level", "periode"]
FEATURES = NUM_COLS + CAT_COLS

print("Baris:", len(df), "| anomali:", int(df[LABEL].sum()), f"({df[LABEL].mean():.2%})")
print("Fitur numerik :", NUM_COLS)
print("Fitur kategori:", CAT_COLS)
df[["employee_id", "gaji_pokok", "tunjangan", "total_gaji", LABEL]].head(3)

Baris: 1800 | anomali: 72 (4.00%)
Fitur numerik : ['gaji_pokok', 'tunjangan', 'lembur_jam', 'tarif_lembur', 'potongan', 'total_gaji', 'rasio_tunjangan', 'rasio_potongan', 'total_per_pokok', 'biaya_lembur']
Fitur kategori: ['departemen', 'level', 'periode']


,employee_id,gaji_pokok,tunjangan,total_gaji,is_anomaly
0,EMP00531,15762000.0,3188000.0,19126400.0,0
1,EMP00315,5297000.0,1462000.0,6655600.0,0
2,EMP00290,11547000.0,2045000.0,13015400.0,0


## 2. EDA — Sebaran Anomali

In [3]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
df[LABEL].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="#E45756")
axes[0].set_title("Jumlah anomali"); axes[0].set_xticklabels(["Normal", "Anomali"], rotation=0)

sns.boxplot(data=df, x=LABEL, y="gaji_pokok", ax=axes[1], showfliers=True)
axes[1].set_yscale("log"); axes[1].set_title("gaji_pokok (log)")

sns.boxplot(data=df, x=LABEL, y="biaya_lembur", ax=axes[2])
axes[2].set_title("biaya lembur")
plt.tight_layout(); plt.show()

print("Anomali per jenis:")
print(df[df[LABEL] == 1]["jenis_anomali"].value_counts().to_string())

Anomali per jenis:
jenis_anomali
gaji_typo                18
potongan_besar           15
tunjangan_negatif        14
lembur_ekstrem           13
total_tidak_konsisten    12


## 3. Aturan Naif — z-score per Kolom

Pembanding sederhana: tandai baris yang punya |z| > 3 pada salah satu kolom
numerik. Sering dipakai di praktik, tapi mudah kewalahan pada data yang
ekornya tebal.

In [4]:
z = (df[NUM_COLS] - df[NUM_COLS].mean()) / df[NUM_COLS].std(ddof=0)
score_z = z.abs().max(axis=1)
pred_z = (score_z > 3).astype(int)

print(f"Anomali terdeteksi aturan z>3: {int(pred_z.sum())}")
print(f"precision: {precision_score(df[LABEL], pred_z, zero_division=0):.4f}")
print(f"recall   : {recall_score(df[LABEL], pred_z):.4f}")
print(f"PR-AUC   : {average_precision_score(df[LABEL], score_z):.4f}")

Anomali terdeteksi aturan z>3: 73


precision: 0.8082
recall   : 0.8194
PR-AUC   : 0.9056


## 4. Unsupervised — Isolation Forest & LOF

Dilatih pada **seluruh data tanpa label**; skor anomali = −score_samples.
Persentase kontaminasi disetel ke laju anomali yang diasumsikan (~4%).

In [5]:
X_num = df[NUM_COLS].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_num)

iso = IsolationForest(n_estimators=400, contamination=0.04, random_state=SEED, n_jobs=-1)
iso.fit(X_scaled)
score_iso = -iso.score_samples(X_scaled)

lof = LocalOutlierFactor(n_neighbors=25, contamination=0.04)
lof.fit(X_scaled)
score_lof = -lof.negative_outlier_factor_

for nama, skor in {"Isolation Forest": score_iso, "LOF": score_lof}.items():
    print(f"{nama:<18} PR-AUC={average_precision_score(df[LABEL], skor):.4f} "
          f"ROC-AUC={roc_auc_score(df[LABEL], skor):.4f}")

Isolation Forest   PR-AUC=0.8709 ROC-AUC=0.9782
LOF                PR-AUC=0.9887 ROC-AUC=0.9975


## 5. Supervised — Random Forest (memakai label)

Ini batas atas: kalau ada label historis, model supervised biasanya menang.
Split stratified 75/25.

In [6]:
X = df[FEATURES]
y = df[LABEL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED)

preprocess = ColumnTransformer([
    ("num", Pipeline([("sc", StandardScaler())]), NUM_COLS),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
])

rf = Pipeline([("prep", preprocess),
               ("clf", RandomForestClassifier(n_estimators=400, min_samples_leaf=2,
                                              class_weight="balanced_subsample",
                                              random_state=SEED, n_jobs=-1))])
rf.fit(X_train, y_train)
score_rf = rf.predict_proba(X_test)[:, 1]

print(f"Random Forest PR-AUC : {average_precision_score(y_test, score_rf):.4f}")
print(f"Random Forest ROC-AUC: {roc_auc_score(y_test, score_rf):.4f}")

Random Forest PR-AUC : 1.0000
Random Forest ROC-AUC: 1.0000


## 6. Evaluasi Deteksi & Precision@k

Untuk tim payroll, yang penting adalah: kalau kita periksa **k transaksi
teratas**, berapa persen yang benar anomali? (kapasitas review terbatas)

In [7]:
# Bandingkan semua metode pada skala yang sama (seluruh data, skor dinormalisasi 0-1)
def norm01(x):
    x = np.asarray(x, dtype=float)
    return (x - x.min()) / (x.max() - x.min() + 1e-12)


skor = {
    "Aturan z-score": norm01(score_z),
    "Isolation Forest": norm01(score_iso),
    "LOF": norm01(score_lof),
}
# RF hanya punya skor di test; evaluasi precision@k dilakukan pada test set terpisah
hasil = pd.DataFrame({
    "metode": list(skor.keys()) + ["Random Forest (test)"],
    "pr_auc": [average_precision_score(df[LABEL], s) for s in skor.values()] +
              [average_precision_score(y_test, score_rf)],
}).sort_values("pr_auc", ascending=False).reset_index(drop=True)
print(hasil.round(4).to_string(index=False))

def precision_at_k(skor_arr, label, k):
    idx = np.argsort(-np.asarray(skor_arr))[:k]
    return float(np.asarray(label)[idx].mean())

k = int(df[LABEL].sum())  # review sebanyak jumlah anomali sebenarnya
print(f"\nPrecision@k (k={k}) pada seluruh data:")
for nama, s in skor.items():
    print(f"  {nama:<18}: {precision_at_k(s, df[LABEL], k):.4f}")

fig, ax = plt.subplots(figsize=(7, 5))
for nama, s in skor.items():
    prec, rec, _ = precision_recall_curve(df[LABEL], s)
    ax.plot(rec, prec, label=f"{nama} (AP={average_precision_score(df[LABEL], s):.3f})")
prec, rec, _ = precision_recall_curve(y_test, score_rf)
ax.plot(rec, prec, label=f"Random Forest (AP={average_precision_score(y_test, score_rf):.3f})",
        color="black", ls="--")
ax.axhline(df[LABEL].mean(), color="grey", ls=":", label="baseline acak")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Kurva Precision–Recall — deteksi anomali payroll"); ax.legend()
plt.tight_layout(); plt.show()

              metode  pr_auc
Random Forest (test)  1.0000
                 LOF  0.9887
      Aturan z-score  0.9056
    Isolation Forest  0.8709

Precision@k (k=72) pada seluruh data:
  Aturan z-score    : 0.8194
  Isolation Forest  : 0.8194
  LOF               : 0.9861


## 7. Contoh Kasus yang Terdeteksi

Menampilkan beberapa anomali dengan skor tertinggi dari Isolation Forest,
beserta label sebenarnya — untuk memverifikasi model "masuk akal".

In [8]:
df_eval = df.copy()
df_eval["skor_iso"] = score_iso
contoh = df_eval.sort_values("skor_iso", ascending=False).head(10)
cols = ["employee_id", "periode", "gaji_pokok", "tunjangan", "lembur_jam",
        "potongan", "total_gaji", "jenis_anomali"]
print(contoh[cols].to_string(index=False))

employee_id periode  gaji_pokok  tunjangan  lembur_jam     potongan    total_gaji  jenis_anomali
   EMP00019 2025-01  26763000.0  4013000.0         384 1.805000e+06  1.180590e+08 lembur_ekstrem
   EMP00159 2025-01  17091000.0  1576000.0         394 1.875000e+06  7.518280e+07 lembur_ekstrem
   EMP00375 2025-01  26029000.0  3945000.0         317 2.223000e+06  9.929790e+07 lembur_ekstrem
   EMP00320 2025-02  20314000.0  2942000.0         383 1.239000e+06  8.946330e+07 lembur_ekstrem
   EMP00103 2025-02  27048000.0  4433000.0           7 3.790195e+07 -4.779449e+06 potongan_besar
   EMP00417 2025-01  22391000.0  6708000.0           5 3.501747e+07 -4.947973e+06 potongan_besar
   EMP00514 2025-03  21691000.0  3093000.0          10 3.401476e+07 -7.349761e+06 potongan_besar
   EMP00451 2025-03  23265000.0  5296000.0          10 3.227811e+07 -1.700108e+06 potongan_besar
   EMP00346 2025-02   8664000.0  1077000.0         379 7.840000e+05  3.741990e+07 lembur_ekstrem
   EMP00579 2025-01  20112000.

## 8. Simpan Artifact

Model unsupervised (scaler + Isolation Forest) disimpan karena tidak butuh
label untuk dipakai ulang pada periode baru.

In [9]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump({"scaler": scaler, "model": iso, "fitur": NUM_COLS},
            ARTIFACT_DIR / "payroll_anomaly_model.joblib")

metadata = {
    "versi_model": "v1",
    "model": "IsolationForest",
    "tujuan": "Deteksi anomali payroll (unsupervised)",
    "fitur_numerik": NUM_COLS,
    "fitur_kategorikal": CAT_COLS,
    "kontaminasi": 0.04,
    "metrik_evaluasi": {
        "isolation_forest_pr_auc": round(float(average_precision_score(df[LABEL], score_iso)), 4),
        "isolation_forest_roc_auc": round(float(roc_auc_score(df[LABEL], score_iso)), 4),
        "lof_pr_auc": round(float(average_precision_score(df[LABEL], score_lof)), 4),
        "zscore_pr_auc": round(float(average_precision_score(df[LABEL], score_z)), 4),
        "random_forest_pr_auc_test": round(float(average_precision_score(y_test, score_rf)), 4),
    },
    "dilatih_pada": pd.Timestamp.utcnow().isoformat(),
    "dataset": str(DATA_PATH),
    "dataset_sintetis": True,
}
(ARTIFACT_DIR / "model_metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
print("Tersimpan:", ARTIFACT_DIR)
print(json.dumps(metadata["metrik_evaluasi"], indent=2))

Tersimpan: /workspace/project/hr-intelligence-system/models/payroll_anomaly/v1
{
  "isolation_forest_pr_auc": 0.8709,
  "isolation_forest_roc_auc": 0.9782,
  "lof_pr_auc": 0.9887,
  "zscore_pr_auc": 0.9056,
  "random_forest_pr_auc_test": 1.0
}


## 9. Kesimpulan

- Supervised (Random Forest) biasanya paling tinggi PR-AUC bila label tersedia.
- Isolation Forest kompetitif **tanpa label** → cocok untuk periode baru.
- Aturan z-score sederhana masih berguna sebagai penyaring cepat, tapi tertinggal.
- Untuk produksi: kirim transaksi skor tertinggi ke tim payroll untuk review
  manusia, jangan blokir otomatis.